# Hansen Ch.22 M-Estimators

**Chapter 22 M-Estimators**

理论推导与**面向初学者的详细注释**见同目录 `Hansen_Ch22_Exercises_Solutions.md`（§0、§1）。

本 notebook：Ex 22.3 四次方损失 vs OLS 的效率比较（蒙特卡洛验证）。

> **写给只学过李子奈/陈强的同学：** m-估计量 = 最小化某个 $\rho$ 的估计量（OLS、MLE 都是特例）。统一的夹心方差 $V=Q^{-1}\Omega Q^{-1}$ 不变——换 $\rho$ 只换 $Q$ 和 $\Omega$ 的计算。
> - OLS：$\rho=e^2/2$，MLE（正态）：$\rho=-\log f$；
> - 四次方损失（22.3）：$\rho=e^4/4$——对大残差惩罚更重，但效率低于 OLS（正态下 $V/V_{OLS}=5/3$）。

## 理论结论的蒙特卡洛验证（无需外部数据）

验证 Ex 22.3：正态误差下四次方损失 $g(u)=u^4/4$ 估计量的方差约为 OLS 的 $5/3$ 倍（因为 OLS 是正态 MLE，最优）。

In [ ]:
import numpy as np
from scipy.optimize import minimize_scalar
rng = np.random.default_rng(22)

# 设定: Y = beta*X + e, e ~ N(0, sigma^2), X ~ N(0,1)
# 四次方损失 g(u)=u^4/4: minimize sum (Y-X*theta)^4 / 4
# 一阶条件: sum (Y_i - X_i*theta)^3 * X_i = 0 (凸问题, 唯一解)

n = 500; reps = 2000; beta_true = 1.0; sigma = 1.0
ols_est, quartic_est = [], []
for r in range(reps):
    X = rng.standard_normal(n)
    e = rng.standard_normal(n) * sigma
    Y = beta_true * X + e
    # OLS: minimize sum (Y-X*theta)^2
    b_ols = np.sum(X * Y) / np.sum(X ** 2)
    ols_est.append(b_ols)
    # 四次方损失: minimize sum (Y-X*theta)^4 / 4 (凸, 用有界优化)
    res = minimize_scalar(lambda th: np.sum((Y - X * th) ** 4),
                          bounds=(b_ols - 2, b_ols + 2), method='bounded')
    quartic_est.append(res.x)

ols_est = np.array(ols_est)
quartic_est = np.array(quartic_est)
ratio = np.var(quartic_est) / np.var(ols_est)
print(f"[Ex 22.3] 正态误差下四次方损失 vs OLS:")
print(f"  OLS 方差     = {np.var(ols_est):.6f}")
print(f"  四次方方差   = {np.var(quartic_est):.6f}")
print(f"  方差比       = {ratio:.4f} (理论 5/3 = {5/3:.4f})")
print(f"  ⇒ 四次方损失比 OLS 效率低约 {ratio:.2f} 倍 (OLS 是正态 MLE, 最优)")